# AttriMIL Video Training on Google Colab

This notebook trains AttriMIL on video polyp classification using dataset stored in Google Drive.

**Requirements:**
- Dataset in Google Drive at: `/content/drive/MyDrive/dataset2`
- GitHub repo cloned to: `/content/drive/MyDrive/AttriMIL`
- GPU runtime (Runtime → Change runtime type → GPU)

## 1. Setup: Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify drive is mounted
import os
print("\nDrive contents:")
!ls -lh /content/drive/MyDrive/ | head -20

## 2. Clone/Update Repository from GitHub

**Option A:** Clone for first time  
**Option B:** Pull latest changes if already cloned

In [ ]:
# Option A: Clone repository (first time only)
# Uncomment if running for the first time

# !cd /content/drive/MyDrive && \
#  git clone https://github.com/viethaa/AttriMIL.git

# Option B: Pull latest changes (if already cloned)
!cd /content/drive/MyDrive/AttriMIL && \
 git pull origin main

print("\n✓ Repository updated")

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision scikit-learn pandas numpy tensorboard

# Verify PyTorch GPU support
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Navigate to Project Directory

In [ ]:
# Change to project directory
import os
os.chdir('/content/drive/MyDrive/AttriMIL')

# Verify we're in the right place
print("Current directory:", os.getcwd())
print("\nProject files:")
!ls -1 *.py | head -10

## 5. Verify Dataset Access

In [ ]:
# Check dataset paths
from config_env import get_env_config, print_config

config = get_env_config()
print_config(config)

# Verify dataset exists
print("\n" + "="*60)
print("Dataset Verification")
print("="*60)

import os

if os.path.exists(config['csv_path']):
    print(f"✓ CSV file found: {config['csv_path']}")
    !head -5 {config['csv_path']}
else:
    print(f"✗ CSV file NOT found: {config['csv_path']}")

if os.path.exists(config['feature_dir']):
    num_features = !ls {config['feature_dir']}/*.pt 2>/dev/null | wc -l
    print(f"\n✓ Feature directory found: {config['feature_dir']}")
    print(f"  Number of feature files: {num_features[0].strip()}")
else:
    print(f"\n✗ Feature directory NOT found: {config['feature_dir']}")

print("\n" + "="*60)

## 6. Start Training

**Important Notes:**
- Training will automatically save checkpoints to Google Drive every 30 minutes
- If disconnected, re-run this cell to resume from last checkpoint
- Colab free tier has ~12 hour runtime limit
- Monitor GPU usage to avoid being disconnected

In [ ]:
# Run training
# This will automatically:
# - Load data from Drive
# - Save checkpoints to Drive every 30 min
# - Resume from last checkpoint if available

!python train_colab.py

## 7. Monitor Training with TensorBoard (Optional)

In [ ]:
# Load TensorBoard
%load_ext tensorboard

# Launch TensorBoard
from config_env import get_env_config
config = get_env_config()

%tensorboard --logdir {config['log_dir']}

## 8. Evaluate Model

In [ ]:
# Load best checkpoint and evaluate on test set
import torch
from models.AttriMIL import AttriMIL
from config_env import get_env_config

config = get_env_config()

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttriMIL(feature_dim=2048, n_classes=3).to(device)

# Load best checkpoint
checkpoint_path = f"{config['save_dir']}/best_latest.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['metrics']['best_val_loss']:.4f}")

# Run evaluation
# !python test_colab.py --checkpoint {checkpoint_path}

## 9. Download Results (Optional)

Download trained models and logs to your local machine

In [ ]:
# Compress results for download
from config_env import get_env_config
config = get_env_config()

!cd {config['save_dir']} && \
 tar -czf /content/attrimil_results.tar.gz .

print("\n✓ Results compressed to /content/attrimil_results.tar.gz")
print("Download from Files panel on the left →")

# Or download directly
from google.colab import files
files.download('/content/attrimil_results.tar.gz')

## Tips for Long Training Sessions

1. **Keep browser tab active**: Colab may disconnect if tab is inactive
2. **Use Colab Pro**: For longer runtimes (24h) and better GPUs
3. **Monitor checkpoints**: Check that periodic saves are working
4. **Push to GitHub**: Periodically commit logs/results to GitHub
5. **Resume training**: If disconnected, just re-run the training cell

## Troubleshooting

**Drive not mounted?**
- Re-run cell 1 and authorize Google Drive access

**Dataset not found?**
- Verify dataset is at `/content/drive/MyDrive/dataset2`
- Check path in `config_env.py`

**Out of memory?**
- Reduce batch size (though it's 1 for MIL)
- Use Colab Pro for more RAM
- Check if other notebooks are running

**Training disconnected?**
- Latest checkpoint saved to Drive
- Just re-run training cell to resume